
# Titanic Data Cleaning and Visualization
Author: _Your Name Here_  
Date: _Update as needed_

This notebook demonstrates a full data processing workflow on the Titanic dataset, including:
- Loading a dataset that contains NULL and intentionally injected garbage values
- Systematic data cleaning and imputation
- Feature engineering
- 3–5 visualization techniques to show the effect of the processing
- Clear documentation throughout

> You can run this notebook on Colab or Kaggle without changes. It uses only `pandas`, `numpy`, and `matplotlib`.



## Objectives
1. Load the Titanic dataset and preview its structure and missingness.  
2. Inject controlled “garbage” values to simulate real-world dirty data.  
3. Clean and impute missing/garbage values with principled methods.  
4. Engineer a useful feature to aid analysis.  
5. Visualize and interpret patterns before and after cleaning.



## Setup
Install and import required libraries.


In [ ]:

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# For consistent plots
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True

print(f'Python: {sys.version.split()[0]}')
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)



## 1. Load the dataset
We will load the Titanic dataset from `seaborn` if available. If `seaborn` is not installed, we will fall back to a local CSV if provided, or you can add your own path.


In [ ]:

def load_titanic():
    try:
        import seaborn as sns
        df = sns.load_dataset('titanic')
        source = 'seaborn.load_dataset("titanic")'
    except Exception as e:
        print('seaborn not available or dataset not found:', e)
        # Fallback: attempt to read a local CSV if placed in the same directory
        # You can upload a 'titanic.csv' compatible with kaggle's titanic format.
        try:
            df = pd.read_csv('titanic.csv')
            source = 'local titanic.csv'
        except Exception as e2:
            raise RuntimeError('No data source available. Install seaborn or provide titanic.csv') from e2
    return df, source

df_raw, data_source = load_titanic()
print('Data source:', data_source)
df_raw.head()



### Inspect structure and NULLs


In [ ]:

print('Shape:', df_raw.shape)
print('\nDTypes:\n', df_raw.dtypes)
print('\nNULL counts:\n', df_raw.isna().sum())
df_raw.describe(include='all').T



## 2. Inject controlled garbage values
To ensure we demonstrate cleaning on realistic noise, we’ll intentionally add some garbage values:
- Insert the string `'N/A'` and `'unknown'` into `embarked`
- Insert impossible ages like `-1` and very large numbers into `age`
- Insert non-numeric strings like `'free'` in `fare`

We keep a copy of the dirty dataset as `df_dirty`.


In [ ]:

rng = np.random.default_rng(42)
df_dirty = df_raw.copy()

# Embarked: sprinkle 'N/A' and 'unknown'
if 'embarked' in df_dirty.columns:
    idx = df_dirty.sample(frac=0.05, random_state=42).index
    df_dirty.loc[idx, 'embarked'] = 'N/A'
    idx2 = df_dirty.sample(frac=0.03, random_state=1).index
    df_dirty.loc[idx2, 'embarked'] = 'unknown'

# Age: inject negatives and overly large values
if 'age' in df_dirty.columns:
    idx3 = df_dirty.sample(frac=0.04, random_state=2).index
    df_dirty.loc[idx3, 'age'] = -1  # impossible
    idx4 = df_dirty.sample(frac=0.02, random_state=3).index
    df_dirty.loc[idx4, 'age'] = 999  # unrealistic

# Fare: inject string noise
if 'fare' in df_dirty.columns:
    idx5 = df_dirty.sample(frac=0.02, random_state=4).index
    df_dirty.loc[idx5, 'fare'] = 'free'  # invalid type

print('Garbage injected. Preview:')
df_dirty.head()



### Missingness and anomaly overview
We visualize missingness by counting NULLs per column, and also flag garbage values we injected.


In [ ]:

def plot_missingness(df, title='Missing values per column'):
    null_counts = df.isna().sum().sort_values(ascending=False)
    ax = null_counts.plot(kind='bar')
    ax.set_title(title)
    ax.set_ylabel('Count of NULLs')
    ax.set_xlabel('Columns')
    plt.tight_layout()
    plt.show()

plot_missingness(df_raw, title='Missing values per column — original')
plot_missingness(df_dirty, title='Missing values per column — with garbage')

# Simple anomaly flags
anomalies = {}
if 'age' in df_dirty.columns:
    anomalies['age_negative'] = int((df_dirty['age'] < 0).fillna(False).sum())
    anomalies['age_gt120'] = int((df_dirty['age'] > 120).fillna(False).sum())
if 'fare' in df_dirty.columns:
    anomalies['fare_non_numeric'] = int(pd.to_numeric(df_dirty['fare'], errors='coerce').isna().sum() - df_dirty['fare'].isna().sum())
if 'embarked' in df_dirty.columns:
    anomalies['embarked_na_strings'] = int(df_dirty['embarked'].isin(['N/A','unknown']).sum())

print('Anomaly summary:', anomalies)



## 3. Cleaning strategy
- **Standardize placeholders**: Map `'N/A'` and `'unknown'` to `NaN` in `embarked`.  
- **Coerce numeric fields**: Convert `fare` to numeric, coercing errors to `NaN`.  
- **Age bounds**: Replace negative and extreme ages with `NaN`.  
- **Imputation**:  
  - `age`: median within groups of `sex` and `pclass`  
  - `embarked`: mode  
  - `fare`: median within `pclass`
- **Type fixes**: Ensure categorical types for nominal columns.


In [ ]:

df = df_dirty.copy()

# 1) Standardize placeholders in 'embarked'
if 'embarked' in df.columns:
    df['embarked'] = df['embarked'].replace({'N/A': np.nan, 'unknown': np.nan})

# 2) Coerce fare to numeric
if 'fare' in df.columns:
    df['fare'] = pd.to_numeric(df['fare'], errors='coerce')

# 3) Age bounds handling
if 'age' in df.columns:
    df.loc[df['age'].lt(0) | df['age'].gt(120), 'age'] = np.nan

# 4) Imputation
# age by sex+pclass median
if set(['age','sex','pclass']).issubset(df.columns):
    df['age'] = df['age'].astype('float')
    df['age'] = df.groupby(['sex','pclass'])['age'].transform(lambda s: s.fillna(s.median()))

# embarked mode
if 'embarked' in df.columns:
    embarked_mode = df['embarked'].mode(dropna=True)
    if len(embarked_mode) > 0:
        df['embarked'] = df['embarked'].fillna(embarked_mode.iloc[0])

# fare median by class
if set(['fare','pclass']).issubset(df.columns):
    df['fare'] = df.groupby('pclass', group_keys=False)['fare'].apply(lambda s: s.fillna(s.median()))

# 5) Type fixes
for col in ['sex', 'class', 'embarked', 'who', 'adult_male', 'alone', 'survived', 'pclass', 'deck', 'embark_town']:
    if col in df.columns:
        # choose appropriate type
        if col in ['survived','pclass','adult_male','alone']:
            # Keep survived/pclass as int if numeric-like, booleans as bool
            if col in ['adult_male','alone']:
                df[col] = df[col].astype('bool', errors='ignore')
            else:
                df[col] = pd.to_numeric(df[col], errors='ignore', downcast='integer')
        else:
            df[col] = df[col].astype('category')

print('Cleaning complete. NULL counts after:')
df.isna().sum()



## 4. Feature engineering
We create `family_size = sibsp + parch + 1` which is often predictive in Titanic analyses.


In [ ]:

if set(['sibsp','parch']).issubset(df.columns):
    df['family_size'] = df['sibsp'].fillna(0) + df['parch'].fillna(0) + 1
else:
    df['family_size'] = np.nan

df[['sibsp','parch','family_size']].head()



## 5. Visualizations
We demonstrate 5 techniques using matplotlib only.

1. **Histogram**: Age distribution after cleaning  
2. **Boxplot**: Fare by passenger class (pclass)  
3. **Bar chart**: Survival rate by sex  
4. **Stacked bar chart**: Survival count by passenger class  
5. **Correlation heatmap**: Numeric feature correlations


In [ ]:

# 1) Histogram: Age distribution
age_clean = df['age'].dropna()
plt.hist(age_clean, bins=20)
plt.title('Age Distribution after Cleaning')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()


In [ ]:

# 2) Boxplot: Fare by pclass
# Prepare data arrays for boxplot in order of class 1,2,3 if present
if 'pclass' in df.columns and 'fare' in df.columns:
    groups = []
    labels = []
    for cls in sorted(df['pclass'].dropna().unique()):
        groups.append(df.loc[df['pclass'] == cls, 'fare'].dropna().values)
        labels.append(str(cls))
    plt.boxplot(groups, labels=labels, showfliers=False)
    plt.title('Fare by Passenger Class')
    plt.xlabel('Pclass')
    plt.ylabel('Fare')
    plt.show()


In [ ]:

# 3) Bar chart: Survival rate by sex
if set(['survived','sex']).issubset(df.columns):
    rates = df.groupby('sex')['survived'].mean().sort_values(ascending=False)
    rates.plot(kind='bar')
    plt.title('Survival Rate by Sex')
    plt.xlabel('Sex')
    plt.ylabel('Survival Rate')
    plt.ylim(0,1)
    plt.tight_layout()
    plt.show()


In [ ]:

# 4) Stacked bar chart: Survival counts by pclass
if set(['survived','pclass']).issubset(df.columns):
    counts = df.pivot_table(index='pclass', columns='survived', values='age', aggfunc='count').fillna(0)
    # Ensure columns are in [0,1] order if available
    cols = [c for c in [0,1] if c in counts.columns]
    ax = counts[cols].plot(kind='bar', stacked=True)
    ax.set_title('Survival Counts by Pclass')
    ax.set_xlabel('Pclass')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()


In [ ]:

# 5) Correlation heatmap of numeric features
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr(numeric_only=True)

fig, ax = plt.subplots()
im = ax.imshow(corr.values, aspect='auto')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)
ax.set_title('Correlation Heatmap (numeric features)')
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()



### Bonus: Compare missingness before vs after cleaning


In [ ]:

def missing_bar(df_in, title):
    null_counts = df_in.isna().sum().sort_values(ascending=False)
    ax = null_counts.plot(kind='bar')
    ax.set_title(title)
    ax.set_xlabel('Columns')
    ax.set_ylabel('Count of NULLs')
    plt.tight_layout()
    plt.show()

missing_bar(df_dirty, 'Missingness — with garbage (before cleaning)')
missing_bar(df, 'Missingness — after cleaning')



## 6. Conclusions
- Injected garbage created visible anomalies in `age`, `fare`, and `embarked` that increased missingness and type issues.  
- Cleaning standardized placeholders, coerced types, and applied grouped imputations that respected data structure.  
- Post-cleaning visualizations showed clearer distributions and interpretable relationships, like higher survival rates for females and class-related fare differences.  
- The workflow here can be adapted to other datasets by changing the injection rules and imputation grouping keys.
